# RAG Pipeline Demo

End-to-end walkthrough: ingest documents → ensemble retrieval → LLM answer → agent tool.

**What this notebook covers:**

1. **Ingest** — load markdown files, chunk them, add to the ensemble retriever
2. **Query** — compare raw retrieval results before and after reranking
3. **RAG chain** — wire retriever + prompt + LLM into a LangChain LCEL chain
4. **Agent tool** — expose the retriever as a tool in a ReAct agent
5. **Introspection** — `get_stats()`, record manager counts, BM25 cache

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from dotenv import load_dotenv
from rich import print  # noqa: F401
from rich.table import Table

assert load_dotenv(verbose=True)
sys.path.append(".")

## 1. Ingest documents

We'll use a handful of `Document` objects as a stand-in for real files.  
In production you'd run `cli rag add-files ./docs/ --retriever persistent`
or call `run_flow_ephemeral(rag_file_ingestion_flow, ...)` for batch ingestion.

In [ ]:
from langchain_core.documents import Document

# Simulated knowledge-base entries
KB_DOCS = [
    Document(
        page_content=(
            "RetrieverFactory creates ManagedRetriever instances from YAML config tags. "
            "Use RetrieverFactory.create('my_tag') to get a ready-to-use retriever."
        ),
        metadata={"source": "retriever_factory.md", "section": "overview"},
    ),
    Document(
        page_content=(
            "ManagedRetriever exposes aquery(query, k, filter) for async similarity search "
            "and aadd_documents(docs) for async ingestion. Sync wrappers query() and "
            "add_documents() are available for CLI and notebook use."
        ),
        metadata={"source": "managed_retriever.md", "section": "api"},
    ),
    Document(
        page_content=(
            "The ensemble retriever combines a dense vector retriever and a BM25 keyword "
            "retriever using Reciprocal Rank Fusion. Configure weights in baseline.yaml."
        ),
        metadata={"source": "ensemble.md", "section": "config"},
    ),
    Document(
        page_content=(
            "BM25DocumentStore persists its index under data/bm25_cache/<config_tag>/. "
            "The bm25_index/ folder holds the bm25s vectorizer; documents.json stores "
            "the original documents with full metadata for accurate retrieval."
        ),
        metadata={"source": "bm25.md", "section": "persistence"},
    ),
    Document(
        page_content=(
            "Record managers deduplicate ingested documents using content hashes. "
            "For persistent Chroma stores a SQLite record manager is auto-created at "
            "data/record_manager/<config_tag>.db when record_manager_url is not set."
        ),
        metadata={"source": "dedup.md", "section": "deduplication"},
    ),
    Document(
        page_content=(
            "The CLI command 'cli rag add-files ./docs/ --retriever persistent' ingests "
            "an entire directory using the Prefect rag_file_ingestion_flow. "
            "Use --force to reprocess unchanged files."
        ),
        metadata={"source": "cli.md", "section": "cli"},
    ),
    Document(
        page_content=(
            "RAGToolFactory wraps a ManagedRetriever in an async LangChain tool. "
            "Configure it in your agent YAML profile under tools: with spec: rag_search."
        ),
        metadata={"source": "tool_factory.md", "section": "agent"},
    ),
    Document(
        page_content=(
            "Metadata filters are passed to the underlying vector store. Example: "
            "await managed.aquery('query', filter={'section': 'api'}). "
            "BM25 retrievers ignore filters since bm25s has no native filter support."
        ),
        metadata={"source": "filters.md", "section": "api"},
    ),
]

print(f"{len(KB_DOCS)} documents ready for ingestion")

In [ ]:
from genai_tk.core.retriever_factory import RetrieverFactory

# Use the default (in-memory Chroma) retriever for this demo
# Switch to "persistent" or "hybrid_ensemble" for production use
managed = RetrieverFactory.create("default")
print("Retriever stats before ingestion:", managed.get_stats())

In [ ]:
# Ingest all documents
ids = await managed.aadd_documents(KB_DOCS)
print(f"Ingested {len(KB_DOCS)} documents. Stats after ingestion:")
print(managed.get_stats())

## 2. Query the retriever

Test retrieval quality before plugging into an LLM.

In [ ]:
TEST_QUERIES = [
    "how do I ingest files from the command line?",
    "what is the difference between BM25 and vector search?",
    "how does deduplication work?",
    "how do I use the retriever as an agent tool?",
]

for query in TEST_QUERIES:
    results = await managed.aquery(query, k=2)
    print(f"\n[bold cyan]Q: {query}[/bold cyan]")
    for i, doc in enumerate(results, 1):
        print(f"  {i}. [{doc.metadata['source']}] {doc.page_content[:100]}…")

In [ ]:
# Metadata filter — restrict to 'api' section
api_docs = await managed.aquery("async query method signature", k=3, filter={"section": "api"})
print(f"Filter section=api → {len(api_docs)} result(s):")
for doc in api_docs:
    print(f"  • {doc.metadata['source']}: {doc.page_content[:80]}…")

## 3. RAG chain (LCEL)

Wire the retriever into a LangChain Expression Language chain:

```
question ──► retriever ──► format_docs ──► prompt ──► LLM ──► answer
```

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from genai_tk.core.llm_factory import get_llm
from genai_tk.core.prompts import def_prompt

llm = get_llm()


def format_docs(docs):
    return "\n\n---\n\n".join(f"[{doc.metadata.get('source', '?')}]\n{doc.page_content}" for doc in docs)


rag_prompt = def_prompt(
    system=(
        "You are a helpful assistant. Answer the user's question using ONLY the provided "
        "context. If the answer is not in the context, say so clearly. Be concise."
    ),
    user="Context:\n{context}\n\nQuestion: {question}",
)

rag_chain = (
    {
        "context": (lambda q: q) | (lambda q: managed.retriever) | format_docs,
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# Direct async query → format → LLM (more control than the LCEL pipe)
async def rag_answer(question: str, k: int = 3) -> str:
    docs = await managed.aquery(question, k=k)
    context = format_docs(docs)
    prompt_val = rag_prompt.invoke({"context": context, "question": question})
    response = await llm.ainvoke(prompt_val)
    return response.content


questions = [
    "How do I ingest files from the command line?",
    "How does deduplication work for persistent stores?",
    "What is RAGToolFactory and how is it used in an agent?",
]

for q in questions:
    print(f"\n[bold]Q:[/bold] {q}")
    answer = await rag_answer(q)
    print(f"[green]A:[/green] {answer}")

## 4. Streaming RAG

Stream the LLM response token by token.

In [ ]:
import sys

question = "Explain how BM25 persistence works and where the index is stored."
docs = await managed.aquery(question, k=3)
context = format_docs(docs)
prompt_val = rag_prompt.invoke({"context": context, "question": question})

print(f"[bold]Q:[/bold] {question}\n[green]A:[/green] ", end="")
async for chunk in llm.astream(prompt_val):
    print(chunk.content, end="", flush=True)
print()

## 5. RAG tool for a ReAct agent

`RAGToolFactory` wraps the retriever in an async LangChain tool that an agent can call.

In [ ]:
from genai_tk.tools.langchain.rag_tool_factory import RAGToolConfig, RAGToolFactory

tool_config = RAGToolConfig(
    retriever="default",
    tool_name="knowledge_search",
    tool_description=(
        "Search the genai-tk knowledge base for information about RetrieverFactory, "
        "ManagedRetriever, RAG pipelines, CLI commands, and agent tools."
    ),
    top_k=3,
)

search_tool = RAGToolFactory(llm).create_tool(tool_config)
print(f"Tool created: {search_tool.name!r}")
print(f"Description: {search_tool.description}")

In [ ]:
# Invoke the tool directly (as an agent would)
result = await search_tool.ainvoke({"query": "how to use the retriever as a CLI command"})
print(result)

In [ ]:
# With a runtime metadata filter
result_filtered = await search_tool.ainvoke(
    {
        "query": "async API methods",
        "filter": '{"section": "api"}',
    }
)
print(result_filtered)

## 6. Plug the tool into a ReAct agent

In [ ]:
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(llm, tools=[search_tool])

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="What CLI command do I use to ingest a directory of files?")]}
)

# Print the final answer
for msg in reversed(response["messages"]):
    if msg.type == "ai" and msg.content:
        print("[bold]Agent answer:[/bold]")
        print(msg.content)
        break

## 7. Introspection

In [ ]:
# Retriever stats
stats = managed.get_stats()
table = Table(title="ManagedRetriever stats")
table.add_column("Property", style="cyan")
table.add_column("Value", style="white")
for k, v in stats.items():
    table.add_row(k, str(v))
print(table)

In [ ]:
# Document count in the underlying vector store
from genai_tk.core.embeddings_store import EmbeddingsStore

es = EmbeddingsStore.create_from_config("default")
print("document_count():", es.document_count())
print("get_stats():", es.get_stats())

In [ ]:
# List all configured retrievers
configs = RetrieverFactory.list_available_configs()
table = Table(title="Available retrievers")
table.add_column("Tag", style="green")
for tag in configs:
    table.add_row(tag)
print(table)

## 8. Cleanup

In [ ]:
success = await managed.adelete_store()
print("Store cleared:", success)